In [ ]:
import math
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from pathlib import Path

# Fallback for `display()` when this runs outside Jupyter/IPython
try:
    display
except NameError:
    def display(x):
        print(x)

# Input file
gpkg_file = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Diff_Census_Heat.gpkg"

# Load and keep NHDA rows
gdf = gpd.read_file(gpkg_file)
gdf_nhda = gdf[gdf["type"] == "NHDA"].copy()

# Requested variables (supports typo variant for footprint column)
requested_columns = [
    "nhda_built_up_ratio",
    "nhda_building_volume_density",
    "nhda_building_density",
    "nhda_avg_building_height",
]

if "nhda_avg_bhilding_footprint" in gdf_nhda.columns:
    footprint_col = "nhda_avg_bhilding_footprint"
elif "nhda_avg_building_footprint" in gdf_nhda.columns:
    footprint_col = "nhda_avg_building_footprint"
else:
    raise KeyError("Weder 'nhda_avg_bhilding_footprint' noch 'nhda_avg_building_footprint' ist in den Daten vorhanden.")

requested_columns.append(footprint_col)

# Convert to numeric
for col in requested_columns:
    gdf_nhda[col] = pd.to_numeric(gdf_nhda[col], errors="coerce")

# Build summary table (statistics as columns)
stats_table = pd.DataFrame(index=requested_columns)
stats_table["mean"] = gdf_nhda[requested_columns].mean()
stats_table["std"] = gdf_nhda[requested_columns].std()
stats_table["min"] = gdf_nhda[requested_columns].min()
stats_table["p10"] = gdf_nhda[requested_columns].quantile(0.10)
stats_table["median"] = gdf_nhda[requested_columns].median()
stats_table["p90"] = gdf_nhda[requested_columns].quantile(0.90)
stats_table["max"] = gdf_nhda[requested_columns].max()

stats_table = stats_table.round(3)

# Prettier row names (for display and LaTeX)
row_name_map = {
    "nhda_built_up_ratio": "Built-up Ratio (%)",
    "nhda_building_volume_density": "Building Volume Density (m$^3$/m$^2$)",
    "nhda_building_density": "Building Density",
    "nhda_avg_building_height": "Average Building Height (m)",
    "nhda_avg_bhilding_footprint": "Average Building Footprint (m^2)",
    "nhda_avg_building_footprint": "Average Building Footprint (m^2)",
}

stats_table = stats_table.rename(index=row_name_map)

# Prettier column names
stats_table = stats_table.rename(columns={
    "mean": "Mean",
    "std": "Std.",
    "min": "Min",
    "p10": "P10",
    "median": "Median",
    "p90": "P90",
    "max": "Max",
})

stats_table.index.name = "Variable"

print("Statistik-Tabelle (nur NHDA):")
display(stats_table)

# LaTeX export (booktabs style)
latex_table = stats_table.to_latex(
    index=True,
    escape=False,
    caption="Descriptive statistics of NHDA morphology variables",
    label="tab:nhda_morphology_stats",
    float_format="%.3f",
)

print("\nLaTeX table:\n")
print(latex_table)

# -----------------------------------------------------------------------------
# Shared map styling helpers for the morphology maps
# -----------------------------------------------------------------------------
TARGET_CRS = "EPSG:25832"
GRID_STEP_M = 50000
MAP_PADDING_M = 10000

# NEW: output folder for the maps (was referenced later but never defined)
OUTPUT_DIR_MORPH = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Maps_Morphology")
OUTPUT_DIR_MORPH.mkdir(parents=True, exist_ok=True)

# NEW: purple sequential colormap (was referenced in plot_morphology_map but never defined)
cm_purple = LinearSegmentedColormap.from_list("purple_seq", ["#f2e9f7", "#dcc7ea", "#a689c9", "#7a4fa8", "#4a1a7a"])

# NEW: metadata describing which morphology variables to map and how to label them
# (was referenced in the final for-loop but never defined)
morphology_vars = {
    "nhda_built_up_ratio": {
        "label": "Built-up Ratio (%)",
        "cbar": "Built-up Ratio (%)",
        "suffix": "built_up_ratio",
    },
    "nhda_building_volume_density": {
        "label": "Building Volume Density (m³/m²)",
        "cbar": "Volume Density (m³/m²)",
        "suffix": "volume_density",
    },
    "nhda_building_density": {
        "label": "Building Density",
        "cbar": "Building Density",
        "suffix": "building_density",
    },
    "nhda_avg_building_height": {
        "label": "Average Building Height (m)",
        "cbar": "Avg. Height (m)",
        "suffix": "avg_height",
    },
    footprint_col: {
        "label": "Average Building Footprint (m²)",
        "cbar": "Avg. Footprint (m²)",
        "suffix": "avg_footprint",
    },
}

def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{int(round(y / 1000))}"))
    ax.tick_params(axis="both", which="major", labelsize=10, length=0, colors="#9B9999")

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker="+", s=24, linewidths=1.0, color="#8a8a8a", alpha=0.85, zorder=4, clip_on=True)

def add_north_arrow(ax):
    ax.annotate('', xy=(0.945, 0.960), xytext=(0.945, 0.880), xycoords='axes fraction', textcoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0), zorder=10)
    ax.text(0.945, 0.972, 'N', transform=ax.transAxes, ha='center', va='bottom', fontsize=14, fontweight='bold', color='#222222', zorder=10)

def add_scale_bar(ax):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0
    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000
    x_start = x1 - span_x * 0.35
    y_start = y0 + span_y * 0.060
    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color="#222222", linewidth=1.3, zorder=8)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color="#222222", linewidth=1.0, zorder=8)
    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, "0", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + segment_len, txt_y, "25", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + bar_len, txt_y, "50 km", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)

def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlabel("Easting (km) - UTM 32N", fontsize=11, color="#555555")
    ax.set_ylabel("Northing (km) - UTM 32N", fontsize=11, color="#555555")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)

# -----------------------------------------------------------------------------
# Load VG250 landkreis data for the map frame
# -----------------------------------------------------------------------------
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"
gdf_lk = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE).to_crs(TARGET_CRS)
gdf_lk = gdf_lk[gdf_lk.geometry.notna()].copy()

# NEW: auto-detect the district-key column used to join gdf_nhda <-> gdf_lk
# (previously hard-coded to "lk_code", which doesn't exist in this dataset).
# Extend DISTRICT_KEY_CANDIDATES if your real column name isn't in this list.
DISTRICT_KEY_CANDIDATES = [
    "lk_code", "AGS", "ags", "AGS_0", "ags_0", "krs_code", "kreis_code",
    "district_id", "lk_ags", "SCHLUESSEL", "schluessel", "KRS", "krs",
]

def _find_key_col(gdf, candidates):
    cols_lower = {c.lower(): c for c in gdf.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

nhda_key_col = _find_key_col(gdf_nhda, DISTRICT_KEY_CANDIDATES)
lk_key_col = _find_key_col(gdf_lk, DISTRICT_KEY_CANDIDATES)

if nhda_key_col is None:
    raise KeyError(
        "Could not find a district-id column in gdf_nhda among "
        f"{DISTRICT_KEY_CANDIDATES}. Actual columns are:\n{sorted(gdf_nhda.columns.tolist())}\n"
        "Add the real column name to DISTRICT_KEY_CANDIDATES or set nhda_key_col manually."
    )
if lk_key_col is None:
    raise KeyError(
        "Could not find a district-id column in gdf_lk among "
        f"{DISTRICT_KEY_CANDIDATES}. Actual columns are:\n{sorted(gdf_lk.columns.tolist())}\n"
        "Add the real column name to DISTRICT_KEY_CANDIDATES or set lk_key_col manually."
    )

print(f"Using '{nhda_key_col}' (gdf_nhda) <-> '{lk_key_col}' (gdf_lk) as the district join key.")

# Standardize both onto a single "lk_code" column, dtype-matched as strings
# (mismatched str/int keys are a common cause of silent all-NaN merges).
gdf_nhda["lk_code"] = gdf_nhda[nhda_key_col].astype(str).str.strip()
gdf_lk["lk_code"] = gdf_lk[lk_key_col].astype(str).str.strip()


def add_labels_top_districts(ax, gdf_plot, values_col="mean_values", std_col=None, top_n=5, offset_m=30000):
    """Label the top N districts with edge placement and duplicate-name disambiguation."""

    required_cols = {values_col, "geometry"}
    if not required_cols.issubset(gdf_plot.columns):
        return

    gdf_labels = gdf_plot[gdf_plot[values_col].notna()].copy()
    if gdf_labels.empty:
        return

    top_districts = gdf_labels.nlargest(top_n, values_col).copy()
    if top_districts.empty:
        return

    bounds = gdf_plot.total_bounds
    minx, miny, maxx, maxy = bounds
    center_x = (minx + maxx) / 2

    frame_x0 = minx - MAP_PADDING_M
    frame_x1 = maxx + MAP_PADDING_M
    frame_y0 = miny - MAP_PADDING_M
    frame_y1 = maxy + MAP_PADDING_M
    frame_height = frame_y1 - frame_y0

    x_left = frame_x0 + 3000
    x_right = frame_x1 - 3000
    y_min = frame_y0 + 6000
    y_max = frame_y1 - 6000
    min_gap = frame_height * 0.09   # minimum spacing between label centres

    name_col = next((c for c in ["GeografischerName_GEN", "GEN", "county", "lk_name", "NAME", "name"] if c in top_districts.columns), None)
    if name_col is None:
        return

    def _norm_name(name):
        n = str(name).strip()
        n = n.replace("Landkreis ", "").replace("Lkr. ", "").replace("Stadtkreis ", "")
        n = n.replace("Kreisfreie Stadt ", "").replace("Landeshauptstadt ", "").replace("Stadt ", "")
        return n.split(",")[0].strip().lower()

    type_by_idx = {}
    grouped = {}
    for idx, row in gdf_labels.iterrows():
        grouped.setdefault(_norm_name(row[name_col]), []).append((idx, row.geometry.area))

    for items in grouped.values():
        if len(items) <= 1:
            continue
        items_sorted = sorted(items, key=lambda x: x[1])
        for i, (idx, _) in enumerate(items_sorted):
            type_by_idx[idx] = "stadt" if i == 0 else "landkreis"

    def _label_text(idx, row):
        raw = str(row[name_col]).strip()
        base = raw.split(",")[0].strip()
        value = row[values_col]

        # Build value string with optional std
        if std_col and std_col in row.index and not pd.isna(row[std_col]):
            val_str = f"{value:.2f} ± {row[std_col]:.2f}"
        else:
            val_str = f"{value:.2f}"

        if idx in type_by_idx:
            prefix = "Stadt" if type_by_idx[idx] == "stadt" else "Lkr."
            return f"{prefix} {base}\n({val_str})"

        raw_low = raw.lower()
        if "landkreis" in raw_low or raw_low.startswith("lkr."):
            return f"Lkr. {base}\n({val_str})"
        if "stadt" in raw_low or "landeshauptstadt" in raw_low:
            return f"Stadt {base}\n({val_str})"
        return f"{base}\n({val_str})"

    points = list(top_districts.representative_point())
    items = [(idx, row, p, _label_text(idx, row)) for (idx, row), p in zip(top_districts.iterrows(), points)]

    left_items = sorted([item for item in items if item[2].x <= center_x], key=lambda item: item[2].y, reverse=True)
    right_items = sorted([item for item in items if item[2].x > center_x], key=lambda item: item[2].y, reverse=True)

    def _slot_positions(n, ref_ys):
        """Place n labels close to their reference y-positions, but spaced at least min_gap apart."""
        center_y = sum(ref_ys) / len(ref_ys)
        total_span = min_gap * (n - 1)
        top = min(center_y + total_span / 2, y_max)
        bottom = max(top - total_span, y_min)
        top = min(bottom + total_span, y_max)
        if n == 1:
            return [max(y_min, min(y_max, center_y))]
        step = (top - bottom) / (n - 1)
        return [top - i * step for i in range(n)]

    def _draw_group(group, side):
        if not group:
            return

        x_text = x_left if side == "left" else x_right
        ha = "left" if side == "left" else "right"
        ref_ys = [item[2].y for item in group]
        slots = _slot_positions(len(group), ref_ys)

        for (idx, row, p, label_text), ty in zip(group, slots):
            ax.annotate(
                label_text,
                xy=(p.x, p.y),
                xycoords="data",
                xytext=(x_text, ty),
                textcoords="data",
                ha=ha,
                va="center",
                fontsize=15,
                color="#1f1f1f",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.6),
                arrowprops=dict(arrowstyle="-", color="#6f6e6e", lw=0.8),
                zorder=6,
                clip_on=False,
            )

    _draw_group(left_items, "left")
    _draw_group(right_items, "right")


def plot_morphology_map(var_name, meta):
    agg = gdf_nhda.groupby("lk_code")[var_name].agg(mean_values="mean", std_values="std").reset_index()
    map_df = gdf_lk.merge(agg, on="lk_code", how="left")

    values = map_df["mean_values"].dropna()
    if values.empty:
        raise ValueError(f"Keine Werte für {var_name} gefunden.")

    vmin = values.min()
    vmax = values.max()
    if vmin == vmax:
        vmax = vmin + 1e-9

    fig, ax = plt.subplots(figsize=(8.8, 9.2))
    add_scientific_frame(ax, gdf_lk)
    map_df.plot(
        ax=ax,
        column="mean_values",
        cmap=cm_purple,
        linewidth=0.35,
        edgecolor="#5a5a5a",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={"color": "#d9d9d9", "edgecolor": "#b7b7b7", "label": "No NHDAs"},
    )

    ax.set_title(f"Mean {meta['label']}", fontsize=18, pad=14)
    add_north_arrow(ax)
    add_scale_bar(ax)

    add_labels_top_districts(ax, map_df, values_col="mean_values", std_col="std_values", top_n=5, offset_m=30000)

    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cm_purple)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label(meta["cbar"], fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    missing_handle = Patch(facecolor="#d9d9d9", edgecolor="#b7b7b7", label="No NHDAs")
    ax.legend(handles=[missing_handle], loc="upper left", frameon=True, framealpha=0.95, facecolor="white", edgecolor="#cccccc", fontsize=10)

    out_file = OUTPUT_DIR_MORPH / f"nhda_morphology_{meta['suffix']}_landkreis_mean.jpg"
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_file}")

for var_name, meta in morphology_vars.items():
    if var_name in gdf_nhda.columns:
        plot_morphology_map(var_name, meta)import math
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from pathlib import Path

# Fallback for `display()` when this runs outside Jupyter/IPython
try:
    display
except NameError:
    def display(x):
        print(x)

# Input file
gpkg_file = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Diff_Census_Heat.gpkg"

# Load and keep NHDA rows
gdf = gpd.read_file(gpkg_file)
gdf_nhda = gdf[gdf["type"] == "NHDA"].copy()

# Requested variables (supports typo variant for footprint column)
requested_columns = [
    "nhda_built_up_ratio",
    "nhda_building_volume_density",
    "nhda_building_density",
    "nhda_avg_building_height",
]

if "nhda_avg_bhilding_footprint" in gdf_nhda.columns:
    footprint_col = "nhda_avg_bhilding_footprint"
elif "nhda_avg_building_footprint" in gdf_nhda.columns:
    footprint_col = "nhda_avg_building_footprint"
else:
    raise KeyError("Weder 'nhda_avg_bhilding_footprint' noch 'nhda_avg_building_footprint' ist in den Daten vorhanden.")

requested_columns.append(footprint_col)

# Convert to numeric
for col in requested_columns:
    gdf_nhda[col] = pd.to_numeric(gdf_nhda[col], errors="coerce")

# Build summary table (statistics as columns)
stats_table = pd.DataFrame(index=requested_columns)
stats_table["mean"] = gdf_nhda[requested_columns].mean()
stats_table["std"] = gdf_nhda[requested_columns].std()
stats_table["min"] = gdf_nhda[requested_columns].min()
stats_table["p10"] = gdf_nhda[requested_columns].quantile(0.10)
stats_table["median"] = gdf_nhda[requested_columns].median()
stats_table["p90"] = gdf_nhda[requested_columns].quantile(0.90)
stats_table["max"] = gdf_nhda[requested_columns].max()

stats_table = stats_table.round(3)

# Prettier row names (for display and LaTeX)
row_name_map = {
    "nhda_built_up_ratio": "Built-up Ratio (%)",
    "nhda_building_volume_density": "Building Volume Density (m$^3$/m$^2$)",
    "nhda_building_density": "Building Density",
    "nhda_avg_building_height": "Average Building Height (m)",
    "nhda_avg_bhilding_footprint": "Average Building Footprint (m^2)",
    "nhda_avg_building_footprint": "Average Building Footprint (m^2)",
}

stats_table = stats_table.rename(index=row_name_map)

# Prettier column names
stats_table = stats_table.rename(columns={
    "mean": "Mean",
    "std": "Std.",
    "min": "Min",
    "p10": "P10",
    "median": "Median",
    "p90": "P90",
    "max": "Max",
})

stats_table.index.name = "Variable"

print("Statistik-Tabelle (nur NHDA):")
display(stats_table)

# LaTeX export (booktabs style)
latex_table = stats_table.to_latex(
    index=True,
    escape=False,
    caption="Descriptive statistics of NHDA morphology variables",
    label="tab:nhda_morphology_stats",
    float_format="%.3f",
)

print("\nLaTeX table:\n")
print(latex_table)

# -----------------------------------------------------------------------------
# Shared map styling helpers for the morphology maps
# -----------------------------------------------------------------------------
TARGET_CRS = "EPSG:25832"
GRID_STEP_M = 50000
MAP_PADDING_M = 10000

# NEW: output folder for the maps (was referenced later but never defined)
OUTPUT_DIR_MORPH = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Comparison_NHDA_RA\Maps_Morphology")
OUTPUT_DIR_MORPH.mkdir(parents=True, exist_ok=True)

# NEW: purple sequential colormap (was referenced in plot_morphology_map but never defined)
cm_purple = LinearSegmentedColormap.from_list("purple_seq", ["#f2e9f7", "#dcc7ea", "#a689c9", "#7a4fa8", "#4a1a7a"])

# NEW: metadata describing which morphology variables to map and how to label them
# (was referenced in the final for-loop but never defined)
morphology_vars = {
    "nhda_built_up_ratio": {
        "label": "Built-up Ratio (%)",
        "cbar": "Built-up Ratio (%)",
        "suffix": "built_up_ratio",
    },
    "nhda_building_volume_density": {
        "label": "Building Volume Density (m³/m²)",
        "cbar": "Volume Density (m³/m²)",
        "suffix": "volume_density",
    },
    "nhda_building_density": {
        "label": "Building Density",
        "cbar": "Building Density",
        "suffix": "building_density",
    },
    "nhda_avg_building_height": {
        "label": "Average Building Height (m)",
        "cbar": "Avg. Height (m)",
        "suffix": "avg_height",
    },
    footprint_col: {
        "label": "Average Building Footprint (m²)",
        "cbar": "Avg. Footprint (m²)",
        "suffix": "avg_footprint",
    },
}

def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{int(round(y / 1000))}"))
    ax.tick_params(axis="both", which="major", labelsize=10, length=0, colors="#9B9999")

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker="+", s=24, linewidths=1.0, color="#8a8a8a", alpha=0.85, zorder=4, clip_on=True)

def add_north_arrow(ax):
    ax.annotate('', xy=(0.945, 0.960), xytext=(0.945, 0.880), xycoords='axes fraction', textcoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0), zorder=10)
    ax.text(0.945, 0.972, 'N', transform=ax.transAxes, ha='center', va='bottom', fontsize=14, fontweight='bold', color='#222222', zorder=10)

def add_scale_bar(ax):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0
    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000
    x_start = x1 - span_x * 0.35
    y_start = y0 + span_y * 0.060
    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color="#222222", linewidth=1.3, zorder=8)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color="#222222", linewidth=1.0, zorder=8)
    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, "0", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + segment_len, txt_y, "25", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + bar_len, txt_y, "50 km", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)

def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlabel("Easting (km) - UTM 32N", fontsize=11, color="#555555")
    ax.set_ylabel("Northing (km) - UTM 32N", fontsize=11, color="#555555")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)

# -----------------------------------------------------------------------------
# Load VG250 landkreis data for the map frame
# -----------------------------------------------------------------------------
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"
gdf_lk = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE).to_crs(TARGET_CRS)
gdf_lk = gdf_lk[gdf_lk.geometry.notna()].copy()

# NEW: auto-detect the district-key column used to join gdf_nhda <-> gdf_lk
# (previously hard-coded to "lk_code", which doesn't exist in this dataset).
# Extend DISTRICT_KEY_CANDIDATES if your real column name isn't in this list.
DISTRICT_KEY_CANDIDATES = [
    "lk_code", "AGS", "ags", "AGS_0", "ags_0", "krs_code", "kreis_code",
    "district_id", "lk_ags", "SCHLUESSEL", "schluessel", "KRS", "krs",
]

def _find_key_col(gdf, candidates):
    cols_lower = {c.lower(): c for c in gdf.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

nhda_key_col = _find_key_col(gdf_nhda, DISTRICT_KEY_CANDIDATES)
lk_key_col = _find_key_col(gdf_lk, DISTRICT_KEY_CANDIDATES)

if nhda_key_col is None:
    raise KeyError(
        "Could not find a district-id column in gdf_nhda among "
        f"{DISTRICT_KEY_CANDIDATES}. Actual columns are:\n{sorted(gdf_nhda.columns.tolist())}\n"
        "Add the real column name to DISTRICT_KEY_CANDIDATES or set nhda_key_col manually."
    )
if lk_key_col is None:
    raise KeyError(
        "Could not find a district-id column in gdf_lk among "
        f"{DISTRICT_KEY_CANDIDATES}. Actual columns are:\n{sorted(gdf_lk.columns.tolist())}\n"
        "Add the real column name to DISTRICT_KEY_CANDIDATES or set lk_key_col manually."
    )

print(f"Using '{nhda_key_col}' (gdf_nhda) <-> '{lk_key_col}' (gdf_lk) as the district join key.")

# Standardize both onto a single "lk_code" column, dtype-matched as strings
# (mismatched str/int keys are a common cause of silent all-NaN merges).
gdf_nhda["lk_code"] = gdf_nhda[nhda_key_col].astype(str).str.strip()
gdf_lk["lk_code"] = gdf_lk[lk_key_col].astype(str).str.strip()


def add_labels_top_districts(ax, gdf_plot, values_col="mean_values", std_col=None, top_n=5, offset_m=30000):
    """Label the top N districts with edge placement and duplicate-name disambiguation."""

    required_cols = {values_col, "geometry"}
    if not required_cols.issubset(gdf_plot.columns):
        return

    gdf_labels = gdf_plot[gdf_plot[values_col].notna()].copy()
    if gdf_labels.empty:
        return

    top_districts = gdf_labels.nlargest(top_n, values_col).copy()
    if top_districts.empty:
        return

    bounds = gdf_plot.total_bounds
    minx, miny, maxx, maxy = bounds
    center_x = (minx + maxx) / 2

    frame_x0 = minx - MAP_PADDING_M
    frame_x1 = maxx + MAP_PADDING_M
    frame_y0 = miny - MAP_PADDING_M
    frame_y1 = maxy + MAP_PADDING_M
    frame_height = frame_y1 - frame_y0

    x_left = frame_x0 + 3000
    x_right = frame_x1 - 3000
    y_min = frame_y0 + 6000
    y_max = frame_y1 - 6000
    min_gap = frame_height * 0.09   # minimum spacing between label centres

    name_col = next((c for c in ["GeografischerName_GEN", "GEN", "county", "lk_name", "NAME", "name"] if c in top_districts.columns), None)
    if name_col is None:
        return

    def _norm_name(name):
        n = str(name).strip()
        n = n.replace("Landkreis ", "").replace("Lkr. ", "").replace("Stadtkreis ", "")
        n = n.replace("Kreisfreie Stadt ", "").replace("Landeshauptstadt ", "").replace("Stadt ", "")
        return n.split(",")[0].strip().lower()

    type_by_idx = {}
    grouped = {}
    for idx, row in gdf_labels.iterrows():
        grouped.setdefault(_norm_name(row[name_col]), []).append((idx, row.geometry.area))

    for items in grouped.values():
        if len(items) <= 1:
            continue
        items_sorted = sorted(items, key=lambda x: x[1])
        for i, (idx, _) in enumerate(items_sorted):
            type_by_idx[idx] = "stadt" if i == 0 else "landkreis"

    def _label_text(idx, row):
        raw = str(row[name_col]).strip()
        base = raw.split(",")[0].strip()
        value = row[values_col]

        # Build value string with optional std
        if std_col and std_col in row.index and not pd.isna(row[std_col]):
            val_str = f"{value:.2f} ± {row[std_col]:.2f}"
        else:
            val_str = f"{value:.2f}"

        if idx in type_by_idx:
            prefix = "Stadt" if type_by_idx[idx] == "stadt" else "Lkr."
            return f"{prefix} {base}\n({val_str})"

        raw_low = raw.lower()
        if "landkreis" in raw_low or raw_low.startswith("lkr."):
            return f"Lkr. {base}\n({val_str})"
        if "stadt" in raw_low or "landeshauptstadt" in raw_low:
            return f"Stadt {base}\n({val_str})"
        return f"{base}\n({val_str})"

    points = list(top_districts.representative_point())
    items = [(idx, row, p, _label_text(idx, row)) for (idx, row), p in zip(top_districts.iterrows(), points)]

    left_items = sorted([item for item in items if item[2].x <= center_x], key=lambda item: item[2].y, reverse=True)
    right_items = sorted([item for item in items if item[2].x > center_x], key=lambda item: item[2].y, reverse=True)

    def _slot_positions(n, ref_ys):
        """Place n labels close to their reference y-positions, but spaced at least min_gap apart."""
        center_y = sum(ref_ys) / len(ref_ys)
        total_span = min_gap * (n - 1)
        top = min(center_y + total_span / 2, y_max)
        bottom = max(top - total_span, y_min)
        top = min(bottom + total_span, y_max)
        if n == 1:
            return [max(y_min, min(y_max, center_y))]
        step = (top - bottom) / (n - 1)
        return [top - i * step for i in range(n)]

    def _draw_group(group, side):
        if not group:
            return

        x_text = x_left if side == "left" else x_right
        ha = "left" if side == "left" else "right"
        ref_ys = [item[2].y for item in group]
        slots = _slot_positions(len(group), ref_ys)

        for (idx, row, p, label_text), ty in zip(group, slots):
            ax.annotate(
                label_text,
                xy=(p.x, p.y),
                xycoords="data",
                xytext=(x_text, ty),
                textcoords="data",
                ha=ha,
                va="center",
                fontsize=15,
                color="#1f1f1f",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.6),
                arrowprops=dict(arrowstyle="-", color="#6f6e6e", lw=0.8),
                zorder=6,
                clip_on=False,
            )

    _draw_group(left_items, "left")
    _draw_group(right_items, "right")


def plot_morphology_map(var_name, meta):
    agg = gdf_nhda.groupby("lk_code")[var_name].agg(mean_values="mean", std_values="std").reset_index()
    map_df = gdf_lk.merge(agg, on="lk_code", how="left")

    values = map_df["mean_values"].dropna()
    if values.empty:
        raise ValueError(f"Keine Werte für {var_name} gefunden.")

    vmin = values.min()
    vmax = values.max()
    if vmin == vmax:
        vmax = vmin + 1e-9

    fig, ax = plt.subplots(figsize=(8.8, 9.2))
    add_scientific_frame(ax, gdf_lk)
    map_df.plot(
        ax=ax,
        column="mean_values",
        cmap=cm_purple,
        linewidth=0.35,
        edgecolor="#5a5a5a",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={"color": "#d9d9d9", "edgecolor": "#b7b7b7", "label": "No NHDAs"},
    )

    ax.set_title(f"Mean {meta['label']}", fontsize=18, pad=14)
    add_north_arrow(ax)
    add_scale_bar(ax)

    add_labels_top_districts(ax, map_df, values_col="mean_values", std_col="std_values", top_n=5, offset_m=30000)

    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cm_purple)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label(meta["cbar"], fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    missing_handle = Patch(facecolor="#d9d9d9", edgecolor="#b7b7b7", label="No NHDAs")
    ax.legend(handles=[missing_handle], loc="upper left", frameon=True, framealpha=0.95, facecolor="white", edgecolor="#cccccc", fontsize=10)

    out_file = OUTPUT_DIR_MORPH / f"nhda_morphology_{meta['suffix']}_landkreis_mean.jpg"
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_file}")

for var_name, meta in morphology_vars.items():
    if var_name in gdf_nhda.columns:
        plot_morphology_map(var_name, meta)